In [1]:
import re
import os
import csv
os.getcwd()

'/Users/angel/Desktop/prism-games/prism-examples/csgs/learning/analysis'

In [2]:
import re
import os
import csv

# ===== INPUT / OUTPUT =====
OUTPUT_DIR = os.getcwd() + "/../results/full/"
OUTPUT_NAME = "full-results.csv"
OUTPUT_FILE = OUTPUT_DIR + OUTPUT_NAME

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
NUM = r"([0-9\.Ee\-]+|NaN|Infinity|-Infinity)"

def to_float(x):
    if not x:
        return ""
    try:
        return float(x)
    except ValueError:
        return ""

HEADER = [
    "case study", "property", 
    "p_reach", "p_T", 
    # "eps", "confidence", 
    "effective horizon",
    "Execution time (ms)", 
    "episodes", 
    "deltaT", "nMin",
    "Robust foundNE", "Robust value", 
    # "Robust true value", 
    "Robust estimation error", 
    "Robust value gap", 
    # "Robust dev gain",
    "Point foundNE", "Point value", 
    # "Point true value", 
    "Point estimation error", 
    "Point value gap", 
    # "Point dev gain",
    "True found", "True coalition results", "True SW value"
]

In [7]:
def parse_file(fname, writer):
    log_file = os.path.join(OUTPUT_DIR, fname)

    # skip directories / non-files
    if not os.path.isfile(log_file):
        return
    
    # ===== READ LOG =====
    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    def extract(pattern, group=1, default=""):
        m = re.search(pattern, text, re.DOTALL)
        return m.group(group).strip() if m else default
    
    # ===== CASE STUDY =====
    property_str = extract(r"Property:\s*(.+)")
    p_reach = extract(rf"Lower bound on reachability probability:\s*{NUM}")
    p_T = extract(rf"Stopping probability:\s*{NUM}")
    eff_horizon = extract(r"Effective horizon:\s*([0-9]+)")
    eps = extract(rf"Epsilon:\s*{NUM}")
    confidence = extract(rf"Confidence:\s*{NUM}")
    
    # ===== GLOBAL =====
    exec_time = extract(r"Execution time:\s*([0-9\.]+)")
    episodes = extract(r"Episodes=([0-9]+)")
    deltaT = extract(rf"DeltaT={NUM}")
    nMin = extract(r"nMin=([0-9]+)")
    
    # ===== TRUE =====
    true_found = extract(r"True SolveOutcome\{found\s*=\s*(true|false)").upper()
    true_value = extract(rf"True SolveOutcome\{{.*?value\s*=\s*{NUM}")
    true_coalition_res = extract(r"Coalition results \(initial state\):\s*(\([0-9\.,]+\))")
    
    # ===== ROBUST =====
    robust_found = extract(r"Robust SolveOutcome\{found\s*=\s*(true|false)").upper()
    robust_value = extract(rf"Robust SolveOutcome\{{.*?value\s*=\s*{NUM}")
    
    robust_true_val = extract(rf"Evaluating robust strategy.*?True value of learned strategy:\s*{NUM}")
    robust_estim_error = extract(rf"Evaluating robust strategy.*?Estimation error:\s*{NUM}")
    robust_gap = extract(rf"Evaluating robust strategy.*?Value gap:\s*{NUM}")
    robust_dev = extract(rf"Evaluating robust strategy.*?Max deviation gain:\s*{NUM}")
    
    # ===== POINT =====
    point_found = extract(r"Point SolveOutcome\{found\s*=\s*(true|false)").upper()
    point_value = extract(rf"Point SolveOutcome\{{.*?value\s*=\s*{NUM}")
    
    point_true_val = extract(rf"Evaluating point strategy.*?True value of learned strategy:\s*{NUM}")
    point_estim_error = extract(rf"Evaluating point strategy.*?Estimation error:\s*{NUM}")
    point_gap = extract(rf"Evaluating point strategy.*?Value gap:\s*{NUM}")
    point_dev = extract(rf"Evaluating point strategy.*?Max deviation gain:\s*{NUM}")
    
    # ===== ROW =====
    row = [
        fname,
        property_str,
    
        to_float(p_reach),
        to_float(p_T),
        # to_float(eps),
        # to_float(confidence),
        int(eff_horizon) if eff_horizon else "",
    
        float(exec_time) if exec_time else "",
        int(episodes) if episodes else "",
        to_float(deltaT),
        int(nMin) if nMin else "",
    
        robust_found,
        to_float(robust_value),
        # to_float(robust_true_val),
        to_float(robust_estim_error),
        to_float(robust_gap),
        # to_float(robust_dev),
    
        point_found,
        to_float(point_value),
        # to_float(point_true_val),
        to_float(point_estim_error),
        to_float(point_gap),
        # to_float(point_dev),
    
        true_found,
        true_coalition_res,
        to_float(true_value)
    ]

    writer.writerow(row)

In [8]:
file_exists = os.path.exists(OUTPUT_FILE)

with open(OUTPUT_FILE, "a", newline="") as f:
    writer = csv.writer(f)

    if not file_exists or os.stat(OUTPUT_FILE).st_size == 0:
        writer.writerow(HEADER)

    # ===== LOOP OVER ALL FILES =====
    for fname in os.listdir(OUTPUT_DIR):
        # skip the output csv itself
        if fname == OUTPUT_NAME or fname.startswith("."):
            continue
        parse_file(fname, writer)
        print("Parsed:", fname)

print("All files processed →", OUTPUT_FILE)

Parsed: delayed_coord2
Parsed: safe_risky1
Parsed: traffic_merge1
Parsed: hide_or_run
Parsed: mixed_ne
Parsed: cyclic_prefs
Parsed: safe_risky3
Parsed: safe_risky2
Parsed: traffic_merge2
All files processed → /Users/angel/Desktop/prism-games/prism-examples/csgs/learning/analysis/../results/full/full-results.csv
